### Ames Housing Linear Regression Project
Group number:

Members:
- My Pham (UFID: 12494292)
- 
- 

**1. Load the Dataset**

In [4]:
from pathlib import Path
import pandas as pd
import os, sys

print("Notebook cwd:", os.getcwd())
name = "AmesHousing.csv"
# Common candidate locations (local, parent, data folders, home)
candidates = [
    Path.cwd() / name,
    Path.cwd().parent / name,
    Path.cwd() / "data" / name,
    Path.cwd().parent / "data" / name,
    Path.cwd().parent.parent / name,
    Path.home() / name,
]

found = None
tried = []
for p in candidates:
    tried.append(str(p))
    if p.exists():
        found = p
        break

# If still not found, do a limited rglob search from cwd and its parents (depth-limited)
if not found:
    for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        try:
            for p in root.rglob(name):
                tried.append(str(p))
                found = p
                break
            if found:
                break
        except Exception:
            pass

if not found:
    msg = "AmesHousing.csv not found. Paths searched:\n" + "\n".join(tried)
    raise FileNotFoundError(msg)

csv_path = found
print("Using file:", csv_path)
df = pd.read_csv(csv_path)
print("Initial shape:", df.shape)
df.head()

Notebook cwd: c:\Users\phaml\Documents\MAD Final Project Housing Dataset\code
Using file: c:\Users\phaml\Documents\MAD Final Project Housing Dataset\data\AmesHousing.csv
Initial shape: (2930, 82)


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


**2. Data Cleaning**

In [5]:
# 1. Drop exact duplicate rows
before_shape = df.shape
df = df.drop_duplicates()
print(f"Dropped duplicates: {before_shape[0] - df.shape[0]} rows")

# 2. Drop columns with more than 50% missing values
missing_pct = df.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.5].index.tolist()
if cols_to_drop:
    print("Dropping columns with >50% missing:", cols_to_drop)
    df = df.drop(columns=cols_to_drop)
else:
    print("No columns with >50% missing to drop")

# 3. Fill numeric NaNs with median, categorical NaNs with mode (or 'Missing')
num_cols = df.select_dtypes(include=['number']).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

for c in num_cols:
    if df[c].isnull().any():
        median = df[c].median()
        df[c] = df[c].fillna(median)

for c in cat_cols:
    if df[c].isnull().any():
        mode = df[c].mode()
        fill = mode[0] if not mode.empty else 'Missing'
        df[c] = df[c].fillna(fill)
    # strip whitespace for object columns
    if df[c].dtype == 'object':
        df[c] = df[c].str.strip()

# 4. Convert object columns with reasonably small cardinality to 'category' dtype
for c in cat_cols:
    try:
        if df[c].nunique(dropna=False) < 200:
            df[c] = df[c].astype('category')
    except Exception:
        pass

print("After cleaning shape:", df.shape)
print("Total remaining missing values:", df.isnull().sum().sum())
df.head()

Dropped duplicates: 0 rows
Dropping columns with >50% missing: ['Alley', 'Mas Vnr Type', 'Pool QC', 'Fence', 'Misc Feature']
After cleaning shape: (2930, 77)
Total remaining missing values: 0


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Lot Shape,Land Contour,Utilities,...,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,IR1,Lvl,AllPub,...,0,0,0,0,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,Reg,Lvl,AllPub,...,0,0,120,0,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,IR1,Lvl,AllPub,...,0,0,0,0,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,Reg,Lvl,AllPub,...,0,0,0,0,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,IR1,Lvl,AllPub,...,0,0,0,0,0,3,2010,WD,Normal,189900


In [6]:
# Save cleaned dataset to data folder
output_path = Path.cwd().parent / "data" / "AmesHousing_cleaned.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.2f} KB")

Cleaned dataset saved to: c:\Users\phaml\Documents\MAD Final Project Housing Dataset\data\AmesHousing_cleaned.csv
File size: 945.25 KB
